In [2]:
import os
from openai import OpenAI

os.environ.get("OPENAI_API_KEY", "")[:7]
API_MODEL = "gpt-5.4-nano"

In [3]:
client = OpenAI()

In [4]:
review = "배송은 빠르고 좋았지만, 제품의 품질이 기대에 미치지 못했습니다. 포장 상태도 다소 불량하여 제품이 손상될까 걱정되었습니다. 다음에는 더 나은 품질의 제품을 기대합니다."

In [18]:
response = client.chat.completions.create(
    model=API_MODEL,
    messages=[
        {
            "role": "user",
            "content": f"이 리뷰 분석해줘. {review}"
        }
    ],
    max_completion_tokens=300,
)

print(response.choices[0].message.content)

리뷰 내용을 핵심만 정리하면 다음과 같습니다.

### 1) 긍정 평가
- **배송 속도**: “빠르고 좋았음”  
→ 배송 만족도는 높은 편입니다.

### 2) 부정 평가(주요 불만)
- **제품 품질**: “기대에 미치지 못함”  
→ 사용/성능/마감 등 전반적인 품질 수준이 아쉬웠다는 의미로 해석됩니다.
- **포장 상태**: “다소 불량”, “손상될까 걱정”  
→ 포장 견고함이 부족해서 파손 위험이 있었다는 점이 불만 포인트입니다.

### 3) 기대/요구 사항(결론)
- “다음에는 더 나은 품질”  
→ 향후 개선을 기대하는 **개선 요청형 리뷰**입니다.

---

원하시면, 이 리뷰를 **판매자 응대용 답변(사과/조치 제안)** 형태로 자연스럽게 다듬어 드릴까요? (예: 품질 검수, 포장 개선, 재발 방지 등 포함)


# JSON Object

In [19]:
from openai import BadRequestError

try:
    # API 요청 http-post 요청
    r = client.chat.completions.create(
        model=API_MODEL,
        messages=[
            {
                "role": "user",
                "content": f"이 리뷰 분석해줘. {review}"
            }
        ],
        response_format={"type": "json_object"},
        max_completion_tokens=300,
    )

    print(r.choices[0].message.content)

except BadRequestError as e:
    print(e.status_code, str(e))

400 Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}


In [23]:
from openai import BadRequestError, APIConnectionError

try:
    r = client.chat.completions.create(
        model=API_MODEL,
        messages=[
            {
                "role": "user",
                "content": f"이 리뷰 분석해줘. {review}"
            }
        ],
        response_format={"type": "json_object"},
        max_completion_tokens=300,
    )

    print(r.choices[0].message.content)

except BadRequestError as e:
    print("요청 오류:", e)

except APIConnectionError as e:
    print("연결 오류:", e)
    print("실제 원인:", repr(e.__cause__))

요청 오류: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}


## JSON_schema

In [ ]:
from openai import BadRequestError, APIConnectionError
schema = {
    "type": "object",
    "properties": {
        "감정": {"type": "string", "description": "감정 중 하나로. 감정: 긍정적, 부정적, 중립적"},
        "별점": {"type": "string", "description": "별점 중 하나로. 별점: 1, 2, 3, 4, 5. 높은게 긍정적"},
        "요약": {"type": "string", "description": "리뷰의 요약"},
    }
}

try:
    r = client.chat.completions.create(
        model=API_MODEL,
        messages=[
            {
                "role": "user",
                "content": f"이 리뷰를 JSON으로 분석해줘. {review}"
            }
        ],
        response_format={"type": "json_schema", "json_schema": schema},
        max_completion_tokens=300,
    )

except BadRequestError:
    data = json.loads(r.choices[0].message.content)
    print(data.keys())

APIConnectionError: Connection error.